## NOTEBOOK 2: XGBoost + SMOTE



### Step 1: Install Required Libraries

In [ ]:
!pip install xgboost imbalanced-learn -q

### Step 2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, f1_score, recall_score, precision_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

### Step 3: Load Dataset

In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

### Step 4: Data Cleaning

In [ ]:
# Convert TotalCharges to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Handle missing values
missing_idx = df[df['TotalCharges'].isnull()].index
for idx in missing_idx:
    if idx > 0:
        df.loc[idx, 'TotalCharges'] = df.loc[idx-1, 'TotalCharges']

# Drop customerID
df = df.drop('customerID', axis=1)

### Step 5: Label Encoding

In [ ]:
le = LabelEncoder()

# Encode target
df['Churn'] = le.fit_transform(df['Churn'])

# Encode categorical features
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
for col in categorical_features:
    df[col] = le.fit_transform(df[col])

### Step 6: Feature Scaling (Normalization for numerical features)

In [ ]:
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
scaler = MinMaxScaler()

df[numerical_cols] = scaler.fit_transform(df[numerical_cols].astype(float))

### Step 7: Check Class Imbalance

In [ ]:
churn_counts = df['Churn'].value_counts()

### Step 8: Split Features and Target

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

### Step 9: Apply SMOTE

In [ ]:
smote = SMOTE(sampling_strategy=1.0, random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

### Step 10: Train-Test Split (70-30)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.30, random_state=42, stratify=y_resampled
)

### Step 11: Train XGBoost Model

In [ ]:
xgb_model = XGBClassifier(
    learning_rate=0.01,
    max_depth=3,
    n_estimators=1000,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.01, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=1000, n_jobs=None,
              num_parallel_tree=None, ...)

### Step 12: Cross-Validation Score

In [ ]:
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=cv, scoring='roc_auc')

### Step 13: Make Predictions

In [ ]:
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

### Step 14: Evaluate Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

### Step 15: Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

### Step 16: Compare WITH vs WITHOUT SMOTE

In [ ]:
# Train XGBoost WITHOUT SMOTE for comparison
X_train_no_smote, X_test_no_smote, y_train_no_smote, y_test_no_smote = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

xgb_no_smote = XGBClassifier(
    learning_rate=0.01,
    max_depth=3,
    n_estimators=1000,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)

xgb_no_smote.fit(X_train_no_smote, y_train_no_smote)
y_pred_no_smote = xgb_no_smote.predict(X_test_no_smote)

# Calculate metrics
acc_no_smote = accuracy_score(y_test_no_smote, y_pred_no_smote)
f1_no_smote = f1_score(y_test_no_smote, y_pred_no_smote)
recall_no_smote = recall_score(y_test_no_smote, y_pred_no_smote)

In [ ]:
print("\n COMPARISON RESULTS:")
print(f"\n{'Metric':<15} {'Without SMOTE':<15} {'With SMOTE':<15} {'Improvement':<15}")
print("="*60)
print(f"{'Accuracy':<15} {acc_no_smote*100:>6.2f}% {accuracy*100:>13.2f}% {(accuracy-acc_no_smote)*100:>12.2f}%")
print(f"{'Recall':<15} {recall_no_smote*100:>6.2f}% {recall*100:>13.2f}% {(recall-recall_no_smote)*100:>12.2f}%")
print(f"{'F1-Score':<15} {f1_no_smote*100:>6.2f}% {f1*100:>13.2f}% {(f1-f1_no_smote)*100:>12.2f}%")

print(f"\nKey Insight: SMOTE improved recall by {(recall-recall_no_smote)*100:.2f}%, meaning we catch {(recall-recall_no_smote)*100:.2f}% more churners!")



 COMPARISON RESULTS:

Metric          Without SMOTE   With SMOTE      Improvement    
Accuracy         80.41%         81.35%         0.94%
Recall           51.87%         87.92%        36.05%
F1-Score         58.43%         82.50%        24.07%

Key Insight: SMOTE improved recall by 36.05%, meaning we catch 36.05% more churners!
